# UFO Sightings: Trends and Shapes

This notebook uses Pandas and Altair to explore reported UFO sightings over time and across countries.

## 1. Load the data

In [4]:
import pandas as pd
import altair as alt

In [5]:
url = "https://raw.githubusercontent.com/UIUC-iSchool-DataViz/is445_data/main/ufo-scrubbed-geocoded-time-standardized-00.csv"

columns = [
    "datetime", "city", "state", "country",
    "shape", "duration_seconds", "duration_text",
    "comments", "date_posted", "latitude", "longitude"
]

df = pd.read_csv(
    url,
    header=None,
    names=columns,
    low_memory=False
)

df.head()

,datetime,city,state,country,shape,duration_seconds,duration_text,comments,date_posted,latitude,longitude
0,10/10/1949 20:30,san marcos,tx,us,cylinder,2700.0,45 minutes,This event took place in early fall around 194...,4/27/2004,29.883056,-97.941111
1,10/10/1949 21:00,lackland afb,tx,NaN,light,7200.0,1-2 hrs,1949 Lackland AFB&#44 TX. Lights racing acros...,12/16/2005,29.384210,-98.581082
2,10/10/1955 17:00,chester (uk/england),NaN,gb,circle,20.0,20 seconds,Green/Orange circular disc over Chester&#44 En...,1/21/2008,53.200000,-2.916667
3,10/10/1956 21:00,edna,tx,us,circle,20.0,1/2 hour,My older brother and twin sister were leaving ...,1/17/2004,28.978333,-96.645833
4,10/10/1960 20:00,kaneohe,hi,us,light,900.0,15 minutes,AS a Marine 1st Lt. flying an FJ4B fighter/att...,1/22/2004,21.418056,-157.803611


## 2. Prepare the data

Extract the year, count sightings by year and shape, and calculate shape percentages within each country.

In [6]:
df["datetime"] = pd.to_datetime(df["datetime"], errors="coerce")
df["year"] = df["datetime"].dt.year

In [7]:
yearly = (
    df[df["year"].between(1990, 2013)]
    .groupby("year")
    .size()
    .reset_index(name="sightings")
)

In [8]:
shape_data = df.dropna(subset=["country", "shape"]).copy()
shape_data["country"] = shape_data["country"].str.upper()
shape_data["shape"] = shape_data["shape"].str.lower()

In [9]:
shape_counts = (
    shape_data.groupby(["country", "shape"])
    .size()
    .reset_index(name="sightings")
)

In [10]:
shape_counts["total"] = shape_counts.groupby("country")["sightings"].transform("sum")
shape_counts["percentage"] = shape_counts["sightings"] / shape_counts["total"] * 100

In [11]:
top_shapes = shape_data["shape"].value_counts().head(10).index
plot2_data = shape_counts[shape_counts["shape"].isin(top_shapes)].copy()

plot2_data.head()

,country,shape,sightings,total,percentage
3,AU,circle,62,527,11.764706
8,AU,disk,50,527,9.487666
10,AU,fireball,34,527,6.451613
12,AU,formation,20,527,3.795066
13,AU,light,119,527,22.580645


## 3. Sightings over time

In [12]:
# create an interval selection for the year range
brush = alt.selection_interval(encodings=["x"])

In [13]:
# main line chart
line = (
    alt.Chart(yearly)
    .mark_line(point=True, color="steelblue")
    .encode(
        x=alt.X("year:Q", title="Year", axis=alt.Axis(format="d")),
        y=alt.Y("sightings:Q", title="Number of Sightings"),
        tooltip=[
            alt.Tooltip("year:Q", title="Year", format="d"),
            alt.Tooltip("sightings:Q", title="Sightings")
        ]
    )
    .properties(
        width=650,
        height=300,
        title="Reported UFO Sightings by Year (1990–2013)"
    )
)

In [14]:
overview = (
    alt.Chart(yearly)
    .mark_area(color="lightsteelblue")
    .encode(
        x=alt.X("year:Q", title="Select Year Range",
                axis=alt.Axis(format="d")),
        y=alt.Y("sightings:Q", title="Sightings")
    )
    .properties(width=650, height=90)
    .add_params(brush)
)

In [15]:
chart1 = alt.vconcat(
    line.encode(
        x=alt.X(
            "year:Q",
            scale=alt.Scale(domain=brush),
            axis=alt.Axis(format="d"),
            title="Year"
        )
    ),
    overview
).resolve_scale(x="independent")

chart1

alt.VConcatChart(...)

## 4. Reported shapes by country

In [16]:
country_dropdown = alt.binding_select(options=["US", "CA", "GB", "AU", "DE"],name="Select Country: ")

country_select = alt.selection_point(fields=["country"],bind=country_dropdown, value="US")

In [17]:
chart2 = (
    alt.Chart(plot2_data)
    .mark_bar(color="teal")
    .encode(
        x=alt.X("percentage:Q",title="Percentage of Country's Sightings (%)"),
        y=alt.Y(
            "shape:N",
            sort="-x",
            title="Reported Shape"
        ),
        tooltip=[
            alt.Tooltip("shape:N", title="Shape"),
            alt.Tooltip("sightings:Q", title="Sightings"),
            alt.Tooltip("percentage:Q", title="Percentage", format=".2f")
        ]
    )
    .add_params(country_select)
    .transform_filter(country_select)
    .properties(width=650,height=350,title="Most Common Reported UFO Shapes by Country"
    )
)

chart2

alt.Chart(...)

## 5. Export

In [18]:
import os

os.makedirs("charts", exist_ok=True)
chart1.save("charts/sightings_over_time.json")
chart2.save("charts/ufo_shapes.json")